# 01 — Residential PM₂.₅ exposure in Helsinki

**Purpose.** Establish the place-based baseline: sample the FMI-ENFUSER PM₂.₅ surface at populated CAFEIN grid cells.

This is the first layer of the proof of concept. It describes ambient PM₂.₅ at populated locations and does **not** yet account for daily mobility or exposure while travelling.


## 1. Environment and CAFEIN sample-data paths


In [ ]:
#pip install -U cafein.sampledata rasterio geopandas matplotlib

In [ ]:
import cafein.sampledata.helsinki as helsinki
import rasterio
import geopandas as gpd

print("Air quality:", helsinki.air_quality)
print("Population grid:", helsinki.population_grid)
print("GTFS:", helsinki.gtfs)

## 2. Inspect the FMI-ENFUSER air-quality raster


In [ ]:
aq_path = helsinki.air_quality

with rasterio.open(aq_path) as src:
    print("CRS:", src.crs)
    print("Bands:", src.count)
    print("Descriptions:")
    for i, desc in enumerate(src.descriptions, start=1):
        print(i, desc)

    print("Bounds:", src.bounds)
    print("Resolution:", src.res)

## 3. Visualize the PM₂.₅ surface

Band 5 is used for PM₂.₅ in the CAFEIN Helsinki sample raster.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

with rasterio.open(aq_path) as src:
    pm25 = src.read(5).astype(float)

    if src.nodata is not None:
        pm25[pm25 == src.nodata] = np.nan

    bounds = src.bounds

extent = [
    bounds.left,
    bounds.right,
    bounds.bottom,
    bounds.top
]

print("PM2.5 min:", np.nanmin(pm25))
print("PM2.5 max:", np.nanmax(pm25))
print("PM2.5 median:", np.nanmedian(pm25))

plt.figure(figsize=(11, 9))

img = plt.imshow(
    pm25,
    extent=extent,
    origin="upper"
)

plt.colorbar(
    img,
    label="PM₂.₅ (µg/m³)"
)

plt.title("FMI-ENFUSER PM₂.₅ — Helsinki Metropolitan Area")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()
plt.show()

## 4. Load the CAFEIN population grid


In [ ]:
import geopandas as gpd
import cafein.sampledata.helsinki as helsinki

pop = gpd.read_file(helsinki.population_grid)

print("CRS:", pop.crs)
print("Rows:", len(pop))
print(pop.columns.tolist())

display(pop.head())

## 5. Sample PM₂.₅ at populated grid-cell centroids


In [ ]:
import rasterio
import numpy as np

# use centroids of population cells
pop_pts = pop.copy()
pop_pts["geometry"] = pop_pts.geometry.centroid

with rasterio.open(helsinki.air_quality) as src:
    raster_crs = src.crs

# match CRS
pop_pts = pop_pts.to_crs(raster_crs)

coords = [
    (geom.x, geom.y)
    for geom in pop_pts.geometry
]

with rasterio.open(helsinki.air_quality) as src:
    pm25_values = [
        x[0]
        for x in src.sample(coords, indexes=5)
    ]

pop_pts["pm25"] = pm25_values

pop_pts.loc[
    (pop_pts["pm25"] < 0) |
    (pop_pts["pm25"] > 1000),
    "pm25"
] = np.nan

display(
    pop_pts[["pm25", "geometry"]].head()
)

## 6. Map residential / populated-location PM₂.₅


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))

pop_pts.plot(
    column="pm25",
    markersize=8,
    legend=True,
    ax=ax
)

ax.set_title(
    "PM₂.₅ at populated 250 m grid cells — Helsinki"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

### Interpretation guardrail

This is a **spatial residential/population exposure proxy**. It does not show where people spend the rest of the day, which destinations they visit, or the concentration they encounter while travelling. Those extensions are handled in Notebooks 02 and 03.
